# Face swap — single-pass Krea generation, NO masking

**No masks, no crop, no stitch, no compositing.** The whole photo goes to Krea2
once, with one instruction, and the model's own render ships exactly as
generated.

This is a different route from the earlier face-swap notebook. That one used
`crop_stitch`, which regenerates a small face crop and pastes it back through a
soft mask -- and that stitch boundary is what produced the ghosted/haloed
headwear. `full_frame` is no better: it builds its own freeze mask and does
LAB-match + feather compositing.

This notebook uses `run_simple_full_body()` with `raw_model=True`, the only
route in this codebase with genuinely zero post-generation compositing:

| stage | status |
|---|---|
| crop / stitch / paste | never runs (different route) |
| `face_refine` (feathered_soft_composite) | **off** via `simple_full_body_face_refine=False` |
| `body_restore` | off via `raw_model` |
| LAB skin wash | off via `raw_model` |
| `skin_repaint` | off via `raw_model` |
| head-scale clamp / procrustes warp | never runs (different route) |

The tradeoff, stated honestly: with no mask, clothing/hair/background are
preserved by the *prompt* and by img2img seeding from the source latent, not by
a structural guarantee. That is the point -- it is what "just Krea generation"
means -- but it is why the prompt below is explicit about what must not change.

**3 cells: Setup → Upload → Run.**

## 1 · Setup

In [ ]:
from pathlib import Path
import subprocess, os

REPO = Path("/content/headswap_V2")
REPO_URL = "https://github.com/malihashar/headswap_V2.git"
BRANCH = "face-swap-no-mask"

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime -> Change runtime type -> GPU, then Run all."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount("/content/drive")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"],
    check=True,
)

os.chdir(REPO)
subprocess.run(["bash", "scripts/setup_colab.sh", "--krea2"], check=True,
               cwd=str(REPO))
print("Setup complete.")


## 2 · Upload your pairs

Two file pickers, each **multi-select** -- pick all your body photos at once,
then all your face photos at once.

Order matters: the Nth body is paired with the Nth face, sorted by filename. The
cell prints the resulting pairing table so you can check it before running
anything. Naming them `01.png, 02.png, ...` on both sides makes this exact.

Files are written to disk immediately and cell 3 reads from disk, so the two
cells share no in-memory state -- restarts and re-runs cannot break it.

> Uses Colab's native uploader rather than `ipywidgets.FileUpload` buttons:
> those render in Colab but their contents often never sync back to the
> kernel, which is why an earlier version silently saved nothing.

In [ ]:
import io
import shutil
from pathlib import Path

from google.colab import files
from PIL import Image

UPLOAD_DIR = Path("/content/face_swap_uploads")
CLEAR_PREVIOUS = True   # wipe earlier uploads so stale pairs cannot leak in

if CLEAR_PREVIOUS and UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("STEP 1 of 2 — select ALL your BODY / scene photos (multi-select)")
body_uploads = files.upload()

print()
print("STEP 2 of 2 — select ALL your FACE / identity photos (multi-select)")
face_uploads = files.upload()

body_names = sorted(body_uploads)
face_names = sorted(face_uploads)

if len(body_names) != len(face_names):
    raise RuntimeError(
        f"Got {len(body_names)} body photo(s) and {len(face_names)} face "
        "photo(s) -- these must match 1:1. Re-run this cell with matching "
        "sets."
    )
if not body_names:
    raise RuntimeError("Nothing was uploaded -- re-run this cell.")

print()
print("Pairing (check this is what you intended):")
for i, (bn, fn) in enumerate(zip(body_names, face_names), start=1):
    Image.open(io.BytesIO(body_uploads[bn])).convert("RGB").save(
        UPLOAD_DIR / f"body_{i:02d}.png"
    )
    Image.open(io.BytesIO(face_uploads[fn])).convert("RGB").save(
        UPLOAD_DIR / f"face_{i:02d}.png"
    )
    print(f"  {i:2d}.  body={bn!r}   <->   face={fn!r}")

print(f"\n{len(body_names)} pair(s) saved to {UPLOAD_DIR}. Now run cell 3.")


## 3 · Run all pairs

Loads the model once, then runs each pair one at a time, showing each result as
it finishes.

Watch the log for these two lines -- they confirm the mask-free path actually
ran:

```
[krea2 body_route] resolved_mode=simple_full_body ...
[krea2 raw_model] body_restore + LAB wash + skin_repaint all DISABLED ...
```

You should **not** see `crop_stitch`, `procrustes`, `feathered_soft_composite`,
`face_refine`, or `skin_harm` anywhere.

In [ ]:
from pathlib import Path
import importlib.util
import sys, os, time

REPO = Path("/content/headswap_V2")
UPLOAD_DIR = Path("/content/face_swap_uploads")

# Reads from DISK, not from cell 2's in-memory widgets. An earlier version
# depended on the widget objects still existing in the kernel, which broke
# whenever the runtime restarted or a stale copy of the cell was run.
pairs, half_filled = [], []
for i in range(1, 14):
    bp = UPLOAD_DIR / f"body_{i:02d}.png"
    fp = UPLOAD_DIR / f"face_{i:02d}.png"
    if not bp.exists() and not fp.exists():
        continue
    if not bp.exists() or not fp.exists():
        half_filled.append(i)
        continue
    pairs.append((i, bp, fp))

if half_filled:
    print(f"Skipping pairs with only one side uploaded: {half_filled}")
if not pairs:
    raise RuntimeError(
        f"No complete pairs found in {UPLOAD_DIR}. Run cell 2 and upload at "
        "least one Body + Face pair (a ✓ appears next to each button once "
        "the file is on disk)."
    )

spec = importlib.util.spec_from_file_location(
    "colab_env", REPO / "scripts" / "colab_env.py"
)
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
colab_env.apply_env(colab_env.default_paths(use_drive=True))
colab_env.ensure_import_path(REPO)

from headswap.config import load_config
from headswap.pipelines.krea2 import Krea2IdentityEditPipeline
from PIL import Image
from IPython.display import display, Markdown

# Full-frame, single-pass prompt. The model sees the WHOLE photo and
# regenerates it in one go -- there is no mask holding anything still, so
# everything that must stay the same has to be said here.
FACE_SWAP_PROMPT = (
    "Change only the face of the person in the first image so it becomes the "
    "face of the person in the second image -- the bone structure, jawline, "
    "brows, eyes, nose, mouth, and facial skin are the second person's. "
    "Everything else in the photograph stays exactly as it already is. "
    "The hair is unchanged: same hairstyle, same hairline, same length, same "
    "colour, same strands -- the first person's own hair, not the second "
    "person's. "
    "Anything worn on the head -- a hat, cap, headscarf, helmet, hood -- is "
    "unchanged: same shape, size, position, colour and material, solid and "
    "opaque, with clean edges. Nothing is added to the head and nothing is "
    "removed from it. "
    "The clothing is unchanged, down to the same folds, collar, buttons and "
    "colours. The body, pose, hands, and the exact position of every limb are "
    "unchanged. The background is unchanged. The camera angle, framing, "
    "lighting, shadows and colour grade are unchanged. "
    "The head stays at exactly the same angle it already is -- same tilt, "
    "same turn, same size on the frame. The expression stays as it is in the "
    "first image: the same mouth, the same smile or absence of one, the same "
    "eye direction. "
    "Take only facial identity from the second image. Do not take its hair, "
    "its clothing, its background, its lighting or its expression. "
    "A photograph, natural skin texture, sharp and realistic."
)

base_cfg = load_config(str(REPO / "configs" / "krea2_identity_edit.yaml"))
cfg = dict(base_cfg)
cfg.update({
    # --- route selection: force the single-pass, mask-free path -----------
    # simple_full_body is picked for a single-face photo when the body route
    # is enabled. Lowering the "enough body visible" floor from 0.38 means a
    # bust/portrait crop takes this route too, instead of falling back to
    # crop_stitch (which masks).
    "enable_body_route": True,
    "simple_path_below_face_frac_min": 0.05,
    "enable_lighting_route": False,

    # --- the prompt (overrides the built-in head-swap text entirely) ------
    "simple_full_body_prompt": FACE_SWAP_PROMPT,

    # --- NO MASKING, NO COMPOSITING --------------------------------------
    # raw_model ships the model's own render: no body_restore, no LAB skin
    # wash, no skin_repaint.
    "simple_full_body_raw_model": True,
    # The one remaining composite in this route is the face_refine pass,
    # which pastes a separately-regenerated face crop back through
    # feathered_soft_composite. That is a mask. Off.
    "simple_full_body_face_refine": False,

    # --- head-swap-specific behaviour that must NOT apply to a face swap --
    # This route normally REPLACES headwear with the donor's hair. A face
    # swap keeps it.
    "simple_full_body_remove_headwear": False,
    # Experimental garment work from the head-swap branch -- all off.
    "simple_full_body_protect_garments": False,
    "skip_skin_clause_when_covered": False,
    "simple_full_body_garment_containment": False,
    "simple_full_body_restore_stripped_garment": False,
})

CACHE_DIR = Path("/content/.cache/headswap_v2")
OUT_DIR = Path("/content/face_swap_results")
for d in (CACHE_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"{len(pairs)} complete pair(s) found on disk. Loading model once, "
      "then running them one at a time.\n")

pipe = Krea2IdentityEditPipeline(cfg=cfg, cache_dir=CACHE_DIR)

t0 = time.perf_counter()
for n, (idx, body_path, face_path) in enumerate(pairs, start=1):
    body = Image.open(body_path).convert("RGB")
    face = Image.open(face_path).convert("RGB")
    pair_out = OUT_DIR / f"pair_{idx:02d}"
    t = time.perf_counter()
    try:
        res = pipe.run(body, face, out_dir=pair_out)
    except Exception as exc:                     # noqa: BLE001
        print(f"Pair {idx} FAILED: {type(exc).__name__}: {exc}")
        continue
    route = (res.meta or {}).get("edit_mode", "?")
    display(Markdown(
        f"### Pair {idx} &nbsp;·&nbsp; {time.perf_counter() - t:.0f}s "
        f"&nbsp;·&nbsp; route=`{route}` &nbsp;·&nbsp; [{n}/{len(pairs)}]"
    ))
    if route != "simple_full_body":
        print(f"  WARNING: pair {idx} did NOT take the mask-free route "
              f"(route={route}). Its result went through compositing.")
    display(res.image)

print(f"\nAll {len(pairs)} pair(s) done in {time.perf_counter() - t0:.0f}s total.")
print(f"Saved under {OUT_DIR}")
